# 🎙️ Open-Source ElevenLabs Studio (F5-TTS + Whisper)
### Мгновенное Zero-Shot клонирование любого голоса по короткому сэмплу (3–10 секунд)
---
> **Возможности:**
> * 🎯 **Zero-Shot Voice Cloning**: Точное копирование тембра, эмоций и пауз любого голоса.
> * 🇷🇺 🇬🇧 **Русский и Английский язык**: Натуральные интонации без роботизированного акцента.
> * 🎙️ **Встроенный Whisper ASR**: Не нужно вручную вбивать слова из сэмпла — система сама распознает речь.
> * 🎛️ **Полнофункциональный WebUI**: Клонирование голоса, генерация диалогов двух людей, подкасты.
> * 📱 **Публичная ссылка gradio.live**: Работает прямо в браузере с компьютера и смартфона.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/markankamaransheogarat-sys/open-elevenlabs-voice-studio/blob/main/Open_ElevenLabs_Voice_Studio_F5_TTS.ipynb)

## 1. 🚀 Установка зависимостей
Нажмите кнопку **Play** на блоке ниже. Установка займет около 40 секунд.

In [ ]:
# @title ⚙️ 1. Установка окружения (Run Setup)
!nvidia-smi
print('📦 Установка F5-TTS и Whisper ASR...')
!pip install -q f5-tts faster_whisper soundfile torchaudio
!apt-get install -y -qq ffmpeg
print('✅ Окружение успешно настроено!')


## 2. 🎛️ Запуск ElevenLabs Web Studio (Gradio UI)
Запустите блок ниже. Через несколько секунд появится ссылка **`Running on public URL: https://xxxx.gradio.live`**.
Перейдите по ней в браузере на телефоне или компьютере.

In [ ]:
# @title 🚀 2. Запуск ElevenLabs Web Studio (Launch WebUI)
print('🎙️ Запуск полнофункциональной студии F5-TTS...')
print('⏳ Модели и нейровокодер автоматически загрузятся с Hugging Face.')
print('👇 Скопируйте ссылку gradio.live из вывода ниже:')

!f5-tts_infer-gradio --port 7860 --share


## 3. 🐍 Прямой синтез через Python (Опционально)
Если вы хотите синтезировать речь прямо из кода без открытия веб-интерфейса:

In [ ]:
# @title 🎙️ 3. Прямой синтез в блокноте (Python Inference)
import torch
import soundfile as sf
from f5_tts.model import DiT
from f5_tts.infer.utils_infer import load_model, load_vocoder, infer_process
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

device = 'cuda' if torch.cuda.is_available() else 'cpu'
dev_name = torch.cuda.get_device_name(0) if device == 'cuda' else 'CPU'
print(f'⚡ Устройство: {device} ({dev_name})')

print('📥 Загрузка Vocos...')
vocoder = load_vocoder(vocoder_name='vocos', device=device)
model_cfg = dict(dim=1024, depth=22, heads=16, ff_mult=2, text_dim=512, conv_layers=4)
ckpt_path = hf_hub_download(repo_id='SWivid/F5-TTS', filename='F5TTS_Base/model_1200000.safetensors')
model = load_model(DiT, model_cfg, ckpt_path=ckpt_path, device=device)

def synthesize(ref_audio_path, ref_text, text_to_speak, speed=1.0):
    final_wave, sr, _ = infer_process(
        ref_audio=ref_audio_path,
        ref_text=ref_text,
        gen_text=text_to_speak,
        model_obj=model,
        vocoder=vocoder,
        speed=speed,
        device=device,
    )
    out_path = 'output_cloned.wav'
    sf.write(out_path, final_wave, sr)
    print(f'✅ Синтез завершен: {out_path}')
    display(Audio(out_path, autoplay=True))
    return out_path

print('✨ Функция synthesize(...) готова к работе!')


## 4. 🌐 Вызов из Telegram-бота или внешних скриптов (REST API)
Используйте библиотеку `gradio_client` для отправки текста на озвучку из любого стороннего проекта:

In [ ]:
# @title 🌐 4. Пример кода для Telegram-бота / Python API
'''
from gradio_client import Client, handle_file

# Подключение к вашей запущенной ссылке xxxx.gradio.live
client = Client('https://xxxxxxxx.gradio.live/')

result = client.predict(
    ref_audio_input=handle_file('sample.wav'),
    ref_text_input='',  # Оставьте пустым для авто-распознавания Whisper
    gen_text_input='Привет! Это тестовое сообщение от бота с клонированным голосом.',
    remove_silence=True,
    cross_fade_duration_slider=0.15,
    speed_slider=1.0,
    api_name='/basic_tts'
)
print('Готовое аудио сохранено в:', result)
'''
